# Fill Missing Targets via Semi-Supervised Clustering

This notebook clusters all linearized sessions using K-Means and propagates target focus scores (ratings) from labeled sessions to unlabeled sessions within each cluster. It also reports if any clusters could not be filled due to a lack of labeled sessions.


In [1]:
from pathlib import Path
import sys
import pandas as pd

MAL_DIR = Path.cwd()
if MAL_DIR.name != 'MAL':
    MAL_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if path.name == 'MAL')

sys.path.insert(0, str(MAL_DIR))

from scripts.fill_missing_targets import fill_missing_targets_via_clustering

INPUT_PATH = MAL_DIR / 'data' / 'processed' / 'linearized_sessions_with_missing_targets.csv'
OUTPUT_PATH = MAL_DIR / 'data' / 'processed' / 'linearized_session_windows.csv'

INPUT_PATH, OUTPUT_PATH


  expr_tree = ast.Expression(body=last_expr, type_ignores=[])


(PosixPath('/home/edf/Documents/GitSynced/SEP4_/MAL/data/processed/linearized_sessions_with_missing_targets.csv'), PosixPath('/home/edf/Documents/GitSynced/SEP4_/MAL/data/processed/linearized_session_windows.csv'))

In [2]:
linearized_df = pd.read_csv(INPUT_PATH, low_memory=False)

# Compare the target distribution for different propagation strategies
for strat in ['mean', 'sample', 'hierarchical']:
    print(f'\n--- Running target propagation strategy: {strat.upper()} ---')
    filled, info = fill_missing_targets_via_clustering(linearized_df, n_clusters=5, strategy=strat)
    print("Rating value counts:")
    print(filled['rating'].value_counts(dropna=False).sort_index())
    
    # We will use 'hierarchical' as the final dataset strategy
    if strat == 'hierarchical':
        filled_df = filled
        info_df = info
        filled_df.to_csv(OUTPUT_PATH, index=False)

# Display the final cluster reports for the hierarchical strategy
reports_list = []
for cluster_name, report in info_df['cluster_reports'].items():
    report['cluster'] = cluster_name
    reports_list.append(report)
pd.DataFrame(reports_list)[['cluster', 'total_sessions', 'labeled_sessions', 'missing_sessions', 'mean_target', 'fill_value_assigned', 'status']]



--- Running target propagation strategy: MEAN ---
Features used for clustering (37): duration_minutes, n_readings, temperature_latest, temperature_mean, temperature_min, temperature_max, temperature_std, temperature_count, temperature_range, humidity_latest, humidity_mean, humidity_min, humidity_max, humidity_std, humidity_count, humidity_range, light_latest, light_mean, light_min, light_max, light_std, light_count, light_range, noise_latest, noise_mean, noise_min, noise_max, noise_std, noise_count, noise_range, co2_latest, co2_mean, co2_min, co2_max, co2_std, co2_count, co2_range

Applying post-propagation rule-based overrides for poor environmental conditions...
  Overrode 331 unlabeled segments to 1.0 due to severe discomfort conditions.
  Overrode 1791 unlabeled segments to 2.0 due to moderate discomfort conditions.

[WARNING] Could not fill targets for clusters: [1, 4] due to lack of labeled data!
Saved rating distribution plot to: /home/edf/Documents/GitSynced/SEP4_/MAL/notebook

  expr_tree = ast.Expression(body=last_expr, type_ignores=[])


,cluster,total_sessions,labeled_sessions,missing_sessions,mean_target,fill_value_assigned,status
0,Cluster 0,2211,1,2210,3.000000,"Filled 2210 via hierarchy: {5.0: 1101, 4.0: 77...",Filled
1,Cluster 1,2277,0,2277,NaN,"Filled 2277 via hierarchy: {3.0: 1529, 4.0: 748}",Filled
2,Cluster 2,2744,121,2623,3.363636,"Filled 2623 via hierarchy: {3.0: 1597, 4.0: 72...",Filled
3,Cluster 3,2030,3,2027,3.333333,"Filled 2027 via hierarchy: {3.0: 1390, 4.0: 57...",Filled
4,Cluster 4,48,0,48,NaN,NaN,Unfillable (Lack of Data)


In [3]:
if info_df['unfillable_clusters']:
    print(f'Warning: Clusters {info_df["unfillable_clusters"]} could not be filled due to lack of labeled data.')
else:
    print('Success: All clusters successfully filled.')


In [4]:
required_model_columns = ['currentTemperature', 'maxTemp', 'minTemp', 'meanTemp', 'rating']
for col in required_model_columns:
    assert col in filled_df.columns, f'Missing model column: {col}'
print('All model compatibility columns verified successfully.')


All model compatibility columns verified successfully.


  expr_tree = ast.Expression(body=last_expr, type_ignores=[])


In [5]:
filled_df['rating'].value_counts(dropna=False).sort_index()


rating
1.0     332
2.0    1771
3.0    4045
4.0    2200
5.0     914
NaN      48
Name: count, dtype: int64

## Visualizations

Below we display the rating distribution and the 2D PCA cluster projection of the final filled dataset.


### Rating Distribution

![Rating Distribution](rating_distribution.png)


### PCA Clusters

![PCA Clusters](pca_clusters.png)


In [6]:
filled_df[['rating', 'currentTemperature', 'maxTemp', 'minTemp', 'meanTemp', 'duration_minutes']].head(20)


,rating,currentTemperature,maxTemp,minTemp,meanTemp,duration_minutes
0,4.0,20.8,20.9,17.7,19.762500,235.000000
1,4.0,20.3,21.0,20.3,20.710638,235.000000
2,2.0,19.8,20.5,19.8,20.219149,235.000000
3,2.0,19.3,19.7,19.3,19.469565,230.016667
4,2.0,19.0,19.3,19.0,19.104762,235.000000
5,2.0,19.4,19.5,18.8,19.128889,235.000000
6,3.0,20.9,20.9,18.9,19.572917,235.000000
7,3.0,19.7,21.1,19.7,20.691489,235.000000
8,4.0,20.0,20.3,19.2,19.866667,235.000000
9,2.0,19.5,20.0,19.5,19.652174,235.000000
